# Renown Combat Lab — v10 (d10 / CE)Rebuilt from v9. Three things changed:1. **Imports CE, not Combatv3.** Every v9 traceback pointed at the d6 tree.2. **Column names resolved once** — v9 crashed on `df["weapon"]` and `df["win_rate"]`   because summary and matchup frames use different prefixes.3. **Gear tables control for retinue.** v9 ranked Gothic Plate last; 71% of Gothic   builds were Levy, so it was measuring the chassis, not the armor.

In [ ]:
# ── 1. CONFIG ────────────────────────────────────────────────────────────────import os, sys, glob, warningsimport numpy as np, pandas as pdwarnings.filterwarnings("ignore")pd.set_option("display.width", 200, "display.max_columns", 60)CE_DIR  = r"C:\Users\Matt\OneDrive\Desktop\Game\CE"RUN_TAG = "d10_foc10_fat22"          # subfolder under CE/lab_outOUT_DIR = os.path.join(CE_DIR, "lab_out", RUN_TAG)# CE bootstrap — puts shims/ ahead of everything so renown_data -> renown_data_d10.sys.path.insert(0, CE_DIR)import ce_pathsce_paths.install(verbose=False)ce_paths.assert_isolated()      # raises if anything resolves out of Combatv3import renown_data as rdimport dice_config as dcimport loadouts as Limport vectorized_combat as vcprint("data   :", rd.VERSION)print("dice   :", dc.describe())print("out    :", OUT_DIR)print("engine :", os.path.basename(vc.__file__))

## 2. Load and normalize

In [ ]:
# ── 2. LOAD ──────────────────────────────────────────────────────────────────def _read(name):    for ext in (".parquet", ".csv"):        p = os.path.join(OUT_DIR, name + ext)        if os.path.exists(p):            return pd.read_parquet(p) if ext == ".parquet" else pd.read_csv(p)    raise FileNotFoundError(f"{name}[.parquet|.csv] not in {OUT_DIR}")summary  = _read("summary")matchups = _read("matchups")# Column resolution. v9 assumed bare names on frames that carry a_/b_ prefixes;# resolve once here and every later cell uses COL[...] instead of a literal.def resolve(df, *candidates):    for c in candidates:        if c in df.columns:            return c    raise KeyError(f"none of {candidates} in {list(df.columns)[:25]}...")COL = {    "name":    resolve(summary, "name", "a_name"),    "wr":      resolve(summary, "win_rate", "a_win_rate", "a_wr", "wr"),    "wr_dec":  resolve(summary, "decisive_win_rate", "a_wr_decisive", "win_rate"),    "retinue": resolve(summary, "retinue", "a_retinue", "a_ret"),    "weapon":  resolve(summary, "weapon", "a_weapon"),    "armor":   resolve(summary, "armor", "a_armor"),    "shield":  resolve(summary, "shield", "a_shield"),    "ranged":  resolve(summary, "ranged", "a_ranged"),    "mpc":     resolve(summary, "military_pursuit_count", "a_military_pursuit_count", "a_mpc", "mpc"),    "tags":    resolve(summary, "tags", "a_tags"),    "pursuits": resolve(summary, "pursuits", "a_pursuits"),}df = summary.copy()df["shield"] = df[COL["shield"]].fillna("None").replace("", "None")print(f"{len(df)} builds | MPC {df[COL['mpc']].min()}-{df[COL['mpc']].max()} "      f"| {len(matchups)} matchups")print("\nresolved columns:")for k, v in COL.items():    print(f"  {k:9} -> {v}")# Pool coverage — anything thin here cannot be ranked later.print("\npool coverage (n < 30 is not rankable):")for field in ("retinue", "armor", "shield", "weapon", "ranged"):    key = "shield" if field == "shield" else COL[field]    vcnt = df[key].fillna("None").value_counts()    thin = ", ".join(f"{k}({v})" for k, v in vcnt.items() if v < 30)    print(f"  {field:8} {len(vcnt):>2} values" + (f"   THIN: {thin}" if thin else ""))

## 3. The story of a battleHeadline numbers only. `weapon_ap` correlation is the single most diagnosticfigure: it was **−0.01** on d6, meaning weapon choice was statistically inert.If the d10 conversion worked it should have moved materially.

In [ ]:
# ── 3. HEADLINE ──────────────────────────────────────────────────────────────import matplotlib.pyplot as pltdef _num(s):    return pd.to_numeric(s, errors="coerce")wr = _num(df[COL["wr"]])# per-build combat stats, pulled from renown_data (not string-matched)def build_stats(r):    ret = rd.RETINUES[r[COL["retinue"]]]    w   = rd.WEAPONS.get(r[COL["weapon"]], {"ap": 0, "init": 0})    sh  = rd.SHIELDS.get(r["shield"] if r["shield"] != "None" else None, {"save_bonus": 0, "init": 0})    arm = rd.ARMORS.get(r[COL["armor"]], {"save": dc.FACES})    return pd.Series({        "to_hit":    ret["to_hit"],        "shaking":   ret["shaking"],        "endurance": ret["endurance"],        "weapon_ap": w["ap"],        "init":      w["init"] + sh["init"],        "eff_save":  arm["save"] - sh["save_bonus"],    })stats = df.apply(build_stats, axis=1)corr = {c: np.corrcoef(_num(stats[c]), wr)[0, 1] for c in stats.columns}corr = dict(sorted(corr.items(), key=lambda kv: -abs(kv[1])))print("WHAT WINS BATTLES (Pearson r with win rate)")for k, v in corr.items():    bar = "#" * int(abs(v) * 30)    print(f"   {k:10} {v:+.3f}  {bar}")print(f"\n   d6 reference: to_hit -0.83 | shaking -0.68 | eff_save +0.17 | weapon_ap -0.01")fig, ax = plt.subplots(1, 2, figsize=(13, 4))ks = list(corr)[::-1]ax[0].barh(ks, [corr[k] for k in ks],           color=["tab:red" if corr[k] < 0 else "tab:blue" for k in ks])ax[0].axvline(0, color="k", lw=1); ax[0].set_title("Stat vs win rate (Pearson r)")ax[1].hist(wr.dropna(), bins=40, color="tab:blue", alpha=.8)ax[1].axvline(0.5, color="gray", ls=":"); ax[1].set_title("Win-rate distribution")plt.tight_layout(); plt.show()

## 4. Gear, controlled for retinueThe v9 failure mode. A gear table that averages raw win rate is really reportingwhich chassis happened to carry that gear. Here every gear value is scored as**mean deviation from its own retinue's mean** — so a positive number means thegear beat what that chassis does on average, whatever chassis it was.

In [ ]:
# ── 4. RETINUE-CONTROLLED GEAR ───────────────────────────────────────────────df["_wr"]   = _num(df[COL["wr"]])df["_ret"]  = df[COL["retinue"]]ret_mean    = df.groupby("_ret")["_wr"].transform("mean")df["_lift"] = df["_wr"] - ret_mean          # deviation from own-chassis baselineTIER_ORD = ["Crude", "Cast", "Wrought", "Forged", "Crafted"]def controlled(field_key, tier_lookup=None, min_n=30):    key = "shield" if field_key == "shield" else COL[field_key]    g = df.groupby(df[key].fillna("None")).agg(        n=("_wr", "size"), wr_raw=("_wr", "mean"),        lift=("_lift", "mean"), lift_sem=("_lift", "sem"),        pct_levy=("_ret", lambda s: (s == "Levy").mean()))    if tier_lookup is not None:        g["tier"] = [tier_lookup.get(i, "-") for i in g.index]        g["tier"] = pd.Categorical(g["tier"], TIER_ORD + ["-"], ordered=True)    g["rankable"] = g["n"] >= min_n    return g.sort_values("lift", ascending=False).round(3)W_TIER = {k: v["tier"] for k, v in rd.WEAPONS.items()}R_TIER = {k: v["tier"] for k, v in rd.RANGED.items()}A_TIER = {k: v["tier"] for k, v in rd.ARMORS.items()}S_TIER = {(k or "None"): (v["tier"] or "-") for k, v in rd.SHIELDS.items()}for label, field, look in [("MELEE", "weapon", W_TIER), ("RANGED", "ranged", R_TIER),                           ("ARMOR", "armor", A_TIER), ("SHIELD", "shield", S_TIER)]:    g = controlled(field, look)    print(f"\n=== {label} (lift = win rate minus own-retinue mean) ===")    print(g[["n", "tier", "wr_raw", "lift", "lift_sem", "pct_levy", "rankable"]].to_string())    bad = g[~g["rankable"]]    if len(bad):        print(f"  NOT RANKABLE (n<30): {', '.join(bad.index.astype(str))}")

## 5. Tier ladder and the sword litmusSwords are the control series: one per tier, minimal keywords, AP stepping by 1.If they don't come out in tier order, the problem isn't the weapon numbers.

In [ ]:
# ── 5. TIER LADDER + SWORD LITMUS ────────────────────────────────────────────gw = controlled("weapon", W_TIER)tier_roll = (gw[gw["rankable"]].groupby("tier", observed=True)             .agg(n_weapons=("n", "size"), builds=("n", "sum"), lift=("lift", "mean")).round(3))print("=== Tier lift (rankable weapons only) ===")print(tier_roll.to_string())print("\n  rising down the tiers = weapon investment converts; flat = it does not\n")SWORDS = ["Farm Tools", "Short Sword", "Arming Sword", "Bastard Sword", "Estoc"]rows = []for s in SWORDS:    if s not in gw.index:        continue    rows.append({"weapon": s, "tier": rd.WEAPONS[s]["tier"], "ap": rd.WEAPONS[s]["ap"],                 "n": int(gw.loc[s, "n"]), "lift": gw.loc[s, "lift"],                 "rankable": bool(gw.loc[s, "rankable"])})lit = pd.DataFrame(rows)print("=== Sword litmus ===")print(lit.to_string(index=False))if len(lit) > 2 and lit["rankable"].all():    ordered = lit["lift"].is_monotonic_increasing    print(f"\n  in tier order: {'YES' if ordered else 'NO — investigate'}")fig, ax = plt.subplots(1, 2, figsize=(13, 4))r = gw[gw["rankable"]].sort_values("lift")ax[0].barh(r.index, r["lift"], xerr=r["lift_sem"],           color=["tab:green" if x > 0 else "tab:red" for x in r["lift"]])ax[0].axvline(0, color="k", lw=1); ax[0].set_title("Melee lift vs own-retinue mean")if len(lit):    ax[1].plot(lit["weapon"], lit["lift"], "o-", color="tab:blue")    ax[1].axhline(0, color="gray", ls=":"); ax[1].set_title("Sword litmus (should rise)")    ax[1].tick_params(axis="x", rotation=30)plt.tight_layout(); plt.show()

## 6. Keyword liftEvery keyword read from `renown_data` tags, not string-matched against buildnames. Each is scored controlled for retinue **and** for MPC, since expensivekeywords ride on expensive builds.

In [ ]:
# ── 6. KEYWORD LIFT ──────────────────────────────────────────────────────────mpc_mean = df.groupby(df[COL["mpc"]])["_wr"].transform("mean")df["_lift2"] = df["_wr"] - (ret_mean + mpc_mean) / 2.0def tags_of(row):    t = set()    for src, key in ((rd.WEAPONS, COL["weapon"]), (rd.ARMORS, COL["armor"]),                     (rd.RANGED, COL["ranged"])):        t |= set(src.get(row[key], {}).get("tags", []) or [])    sh = row["shield"]    t |= set(rd.SHIELDS.get(None if sh == "None" else sh, {}).get("tags", []) or [])    raw = row[COL["tags"]]    if isinstance(raw, str):        t |= {x.strip() for x in raw.strip("[]{}").replace("'", "").split(",") if x.strip()}    return ttagsets = df.apply(tags_of, axis=1)all_tags = sorted({t for s in tagsets for t in s})rows = []for t in all_tags:    has = tagsets.apply(lambda s: t in s)    if has.sum() < 30 or (~has).sum() < 30:        continue    rows.append({"keyword": t, "n": int(has.sum()),                 "lift": df.loc[has, "_lift2"].mean() - df.loc[~has, "_lift2"].mean()})kw = pd.DataFrame(rows).sort_values("lift", ascending=False).round(3)print("=== Keyword lift (controlled for retinue + MPC, n>=30 both arms) ===")print(kw.to_string(index=False))

## 7. Design targetsThe pass/fail block. MPC↔win-rate correlation was **+0.65** on d6 and is theheadline health metric: if investment doesn't buy wins, nothing else matters.

In [ ]:
# ── 7. DESIGN TARGETS ────────────────────────────────────────────────────────from scipy.stats import pearsonr, spearmanrmpc = _num(df[COL["mpc"]]); w = df["_wr"]ok = mpc.notna() & w.notna()pr, _ = pearsonr(mpc[ok], w[ok]); sr, _ = spearmanr(mpc[ok], w[ok])per_mpc = df.groupby(mpc)["_wr"].agg(["size", "mean", "median", "max"]).round(3)mono = per_mpc["mean"].is_monotonic_increasingret = df.groupby("_ret")["_wr"].mean().sort_values()checks = [    ("MPC <-> win rate (Pearson)", pr, "> 0.60", pr > 0.60),    ("MPC <-> win rate (Spearman)", sr, "> 0.60", sr > 0.60),    ("mean WR monotonic in MPC", float(mono), "True", mono),    ("retinue spread", ret.max() - ret.min(), "< 0.45", (ret.max() - ret.min()) < 0.45),]print("=== DESIGN TARGETS ===")for name, val, target, passed in checks:    print(f"  [{'PASS' if passed else 'FAIL'}] {name:32} {val:+.3f}   target {target}")print(f"\n  retinue means: " + ", ".join(f"{k} {v:.3f}" for k, v in ret.items()))print("\n=== Per-MPC ===")print(per_mpc.to_string())fig, ax = plt.subplots(figsize=(9, 4))ax.plot(per_mpc.index, per_mpc["mean"], "o-", label="mean")ax.plot(per_mpc.index, per_mpc["median"], "^-", label="median")ax.plot(per_mpc.index, per_mpc["max"], "o-", color="tab:red", label="best build")ax.axhline(0.5, color="gray", ls=":"); ax.set_xlabel("MPC"); ax.set_ylabel("win rate")ax.legend(); ax.set_title("Investment curve"); plt.tight_layout(); plt.show()

## 8. Live engine probesRuns against the CE engine directly. Use this to confirm a mechanic fires beforereading anything into a tournament table — v9 reported Destroy Shield at 0.0%while the mechanic was working fine in CE.

In [ ]:
# ── 8. LIVE PROBES ───────────────────────────────────────────────────────────def probe(ret="Sergeant", weapon="Arming Sword", shield=None, armor="Chainmail",          ranged=None, tags=(), size=25):    return L.Loadout(name="probe", retinue=ret, weapon=weapon, shield=shield, armor=armor,                     ranged=ranged, has_tiltyard=False, size=size,                     extra_tags=frozenset(tags), upkeep_per_retinue=0)def duel(a, b, n=4000, seed=7):    r = vc.run_matchup_vec(a, b, n_runs=n, seed=seed)    tot = r["a_wins"] + r["b_wins"] + r["mut_wipe"] + r["indecisive"]    return {"a_wr": r["a_wins"] / tot, "skirm": r["avg_skirm"],            "a_rem": r["avg_a_rem"], "b_rem": r["avg_b_rem"],            "b_shield_destroyed": r["b_shield_destroyed_rate"]}print("=== Destroy Shield fires? (Morningstar vs shields) ===")atk = probe(weapon="Morningstar")for sh in ["Kite Shield", "Tower Shield", "Heater Shield"]:    d = duel(atk, probe(shield=sh))    print(f"  vs {sh:14} destroyed {d['b_shield_destroyed']:.3f}   "          f"{'<- Heater immune, expect 0' if sh == 'Heater Shield' else ''}")print("\n=== Focused threshold honoured? (Cleave carrier vs plain, same AP) ===")for tag, lbl in [((), "base (Crit = FOCUSED_THR)"), (("Crit 9",), "Crit 9"), (("Crit 8",), "Crit 8")]:    a = probe(weapon="Battle Axe", tags=tag)      # Battle Axe carries Cleave    print(f"  {lbl:26} a_wr {duel(a, probe())['a_wr']:.3f}")print("\n=== Improved Parry wired? ===")for tag, lbl in [((), "no Parry"), (("Parry",), "Parry"), (("Improved Parry",), "Improved Parry")]:    print(f"  {lbl:16} defender a_wr {duel(probe(), probe(tags=tag))['a_wr']:.3f}  (lower = better defence)")print("\n=== AP ladder converts? (sword litmus, head to head vs a fixed wall) ===")wall = probe(shield="Tower Shield", armor="Full Plate")for s in ["Farm Tools", "Short Sword", "Arming Sword", "Bastard Sword", "Estoc"]:    d = duel(probe(weapon=s), wall)    print(f"  {s:15} AP {rd.WEAPONS[s]['ap']:>3}   a_wr {d['a_wr']:.3f}   skirm {d['skirm']:.2f}")